In [0]:
%run ../00-common/01.environment-config

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F


LOOKBACK     = 5        # seasons of history to use
ROOKIE_FACTOR = 0.70   # rookies score at 70% of their constructor's pace-setter
DRIVER_W     = 0.55    # personal driver history weight
TEAM_W       = 0.45    # constructor/car quality weight

# Exponential decay: 2025=1.0, 2024=0.70, 2023=0.49 ...
# Computed dynamically after we know the last available season

In [0]:
# Load from silver lineup table
lineup = spark.table(f"{CATALOG}.{SILVER}.season_2026_grid").toPandas()

print(f"2026 Grid  : {len(lineup)} drivers across {lineup['constructor_name'].nunique()} teams")
print(f"Rookies    : {lineup[lineup['driver_rookie']]['driver_name'].tolist()}")
print()
display(spark.table(f"{CATALOG}.{SILVER}.season_2026_grid"))

In [0]:
# Auto-detect last available season so the model stays correct if data is updated later
max_season_row = spark.sql(f"SELECT MAX(season) AS s FROM {CATALOG}.{GOLD}.v_driver_standing").collect()[0]
LAST_SEASON    = int(max_season_row['s'])
START_SEASON   = LAST_SEASON - LOOKBACK + 1

# Exponential decay weights (most recent = 1.0)
SEASON_WEIGHTS = {yr: round(0.70 ** (LAST_SEASON - yr), 4)
                  for yr in range(START_SEASON, LAST_SEASON + 1)}
SEASON_WEIGHTS[LAST_SEASON] = 1.0

driver_hist = spark.sql(f"""
    SELECT driver_name, season, standing, race_starts,
           total_points, number_of_wins, number_of_podiums
    FROM   {CATALOG}.{GOLD}.v_driver_standing
    WHERE  season BETWEEN {START_SEASON} AND {LAST_SEASON}
""").toPandas()

constructor_hist = spark.sql(f"""
    SELECT constructor_name, season, standing, race_starts,
           total_points, number_of_wins, number_of_podiums
    FROM   {CATALOG}.{GOLD}.v_constructor_standing
    WHERE  season BETWEEN {START_SEASON} AND {LAST_SEASON}
""").toPandas()

print(f"Driver records      : {len(driver_hist)}")
print(f"Constructor records : {len(constructor_hist)}")
print("\nConstructor names in recent history:")
for c in sorted(constructor_hist['constructor_name'].unique()):
    print(f"  {c}")

In [0]:
# Map each 2026 team to its historical name(s) in v_constructor_standing
# (Teams often renamed: AlphaTauri→RB, Alfa Romeo→Kick Sauber, Renault→Alpine)
# Keys = silver constructor_name values; values = historical names in v_constructor_standing
CONSTRUCTOR_MAP = {
    "Ferrari":          ["Ferrari"],
    "McLaren":          ["McLaren"],
    "Red Bull":         ["Red Bull"],
    "Mercedes":         ["Mercedes"],
    "Aston Martin":     ["Aston Martin", "Racing Point"],
    "Williams":         ["Williams"],
    "Alpine F1 Team":   ["Alpine F1 Team", "Renault"],
    "Haas F1 Team":     ["Haas F1 Team"],
    "Audi":             ["Kick Sauber", "Alfa Romeo"],   # Sauber rebranded to Audi for 2026
    "RB F1 Team":       ["RB", "AlphaTauri"],
    "Cadillac F1 Team": [],   # Brand-new entry — median fallback applied in scoring cell
}

def weighted_score(df, group_col):
    """Compute exponentially-weighted performance metrics per entity."""
    df = df.copy()
    df['w']           = df['season'].map(SEASON_WEIGHTS).fillna(0.05)
    df['pts_per_race']= df['total_points']     / df['race_starts'].clip(lower=1)
    df['win_rate']    = df['number_of_wins']    / df['race_starts'].clip(lower=1)
    df['podium_rate'] = df['number_of_podiums'] / df['race_starts'].clip(lower=1)
    rows = []
    for name, g in df.groupby(group_col):
        W = g['w'].sum()
        rows.append({
            group_col:         name,
            'w_pts_per_race': (g['pts_per_race']  * g['w']).sum() / W,
            'w_win_rate':     (g['win_rate']       * g['w']).sum() / W,
            'w_podium_rate':  (g['podium_rate']    * g['w']).sum() / W,
            'best_standing':   g['standing'].min(),
            'latest_standing': g.sort_values('season').iloc[-1]['standing'],
            'seasons_active':  len(g),
        })
    return pd.DataFrame(rows)

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

# Driver features
drv_feat = weighted_score(driver_hist, 'driver_name')
drv_feat['drv_score'] = (
    0.60 * minmax(drv_feat['w_pts_per_race']) +
    0.20 * minmax(drv_feat['w_win_rate'])     +
    0.20 * minmax(drv_feat['w_podium_rate'])
)

# Constructor features
con_rows = []
for team26, hist_names in CONSTRUCTOR_MAP.items():
    sub = constructor_hist[constructor_hist['constructor_name'].isin(hist_names)].copy()
    if sub.empty:
        print(f"No historical data found for {team26} (looked for {hist_names})")
        continue
    sub['w']           = sub['season'].map(SEASON_WEIGHTS).fillna(0.05)
    sub['pts_per_race']= sub['total_points']     / sub['race_starts'].clip(lower=1)
    sub['win_rate']    = sub['number_of_wins']    / sub['race_starts'].clip(lower=1)
    sub['podium_rate'] = sub['number_of_podiums'] / sub['race_starts'].clip(lower=1)
    W = sub['w'].sum()
    con_rows.append({
        'constructor_2026': team26,
        'w_pts_per_race':  (sub['pts_per_race']  * sub['w']).sum() / W,
        'w_win_rate':      (sub['win_rate']       * sub['w']).sum() / W,
        'w_podium_rate':   (sub['podium_rate']    * sub['w']).sum() / W,
        'best_standing':    sub['standing'].min(),
        'latest_standing':  sub.sort_values('season').iloc[-1]['standing'],
    })
con_feat = pd.DataFrame(con_rows)
con_feat['con_score'] = (
    0.60 * minmax(con_feat['w_pts_per_race']) +
    0.20 * minmax(con_feat['w_win_rate'])     +
    0.20 * minmax(con_feat['w_podium_rate'])
)

print("Driver feature rows     :", len(drv_feat))
print("Constructor feature rows:", len(con_feat))
print("\nTeam prediction scores (normalized 0-1):")
print(con_feat.sort_values('con_score', ascending=False)[['constructor_2026','con_score','latest_standing']].to_string(index=False))

In [0]:
import unicodedata

def strip_accents(s):
    """Remove accents for name matching (Pérez→Perez, Hülkenberg→Hulkenberg)."""
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s))
                   if not unicodedata.combining(c))

pred = lineup.copy()
pred['_drv_key']      = pred['driver_name'].apply(strip_accents)
drv_feat['_drv_key']  = drv_feat['driver_name'].apply(strip_accents)

# Merge driver score (accent-insensitive)
pred = pred.merge(drv_feat[['_drv_key', 'drv_score']], on='_drv_key', how='left')

# Merge constructor score
pred = pred.merge(con_feat[['constructor_2026', 'con_score']],
                  left_on='constructor_name', right_on='constructor_2026', how='left')

pred['drv_score'] = pred['drv_score'].fillna(0.0)
# Brand-new teams (Cadillac) get the median constructor score as a conservative estimate
median_con = con_feat['con_score'].median()
pred['con_score'] = pred['con_score'].fillna(median_con)
pred.drop(columns=['_drv_key'], inplace=True)

# Composite: rookies have no personal history → rely on constructor + apply discount
pred['composite'] = np.where(
    pred['driver_rookie'],
    pred['con_score'] * ROOKIE_FACTOR,
    DRIVER_W * pred['drv_score'] + TEAM_W * pred['con_score']
)

# Scale to 0–100 for readability
pred['pred_score'] = (pred['composite'] / pred['composite'].max() * 100).round(2)
pred = pred.sort_values('pred_score', ascending=False).reset_index(drop=True)
pred['pred_rank']  = pred.index + 1

missing = pred[(pred['drv_score'] == 0) & (~pred['driver_rookie'])]['driver_name'].tolist()
if missing:
    print(f"⚠  No personal history found for (using constructor score only): {missing}")
else:
    print("✓  All drivers matched to historical data")

In [0]:
wdc = pred[['pred_rank','driver_name','constructor_name','pred_score','driver_rookie']].copy()
wdc.columns = ['Pos', 'Driver', 'Team', 'Prediction Score', 'Rookie']


print("2026 PREDICTED WORLD DRIVERS CHAMPIONSHIP")
display(wdc)

In [0]:
# WCC = sum of both drivers' prediction scores per team
wcc = (
    pred.groupby('constructor_name')['pred_score']
    .agg(['sum', 'max', 'min'])
    .reset_index()
    .rename(columns={
        'constructor_name': 'Team',
        'sum': 'Team Score',
        'max': 'Best Driver Score',
        'min': 'Second Driver Score'
    })
    .sort_values('Team Score', ascending=False)
    .reset_index(drop=True)
)
wcc['Pos']               = wcc.index + 1
wcc['Team Score']        = wcc['Team Score'].round(1)
wcc['Best Driver Score'] = wcc['Best Driver Score'].round(1)
wcc['Second Driver Score'] = wcc['Second Driver Score'].round(1)
wcc = wcc[['Pos','Team','Team Score','Best Driver Score','Second Driver Score']]
display(wcc)

In [0]:


# ── pred_2026_drivers — one row per driver ────────────────────────────────────
wdc_save = (
    pred[['pred_rank', 'driver_name', 'constructor_name', 'driver_rookie',
          'constructor_rookie', 'drv_score', 'con_score', 'pred_score']]
    .rename(columns={'pred_rank': 'predicted_pos', 'pred_score': 'prediction_score'})
)
(spark.createDataFrame(wdc_save)
     .write.format("delta").mode("overwrite")
     .saveAsTable(f"{CATALOG}.{GOLD}.pred_2026_drivers"))

# ── pred_2026_constructors — one row per team ─────────────────────────────────
wcc_save = (
    pred.groupby('constructor_name', as_index=False)
    .agg(
        team_score          = ('pred_score', 'sum'),
        best_driver_score   = ('pred_score', 'max'),
        second_driver_score = ('pred_score', 'min'),
    )
    .sort_values('team_score', ascending=False)
    .reset_index(drop=True)
)
wcc_save['predicted_pos'] = wcc_save.index + 1
wcc_save = wcc_save.round({'team_score': 2, 'best_driver_score': 2, 'second_driver_score': 2})
wcc_save = wcc_save.rename(columns={'constructor_name': 'team'})

(spark.createDataFrame(wcc_save)
     .write.format("delta").mode("overwrite")
     .saveAsTable(f"{CATALOG}.{GOLD}.pred_2026_constructors"))

print(f"\n✓  {CATALOG}.{GOLD}.pred_2026_drivers       — {len(wdc_save)} rows")
print(f"✓  {CATALOG}.{GOLD}.pred_2026_constructors  — {len(wcc_save)} rows")